## 1. Install and import libraries

If imports fail, install the packages in your notebook environment with `pip install pandas numpy scikit-learn matplotlib seaborn jupyter`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)


In [ ]:
%pip install scikit-learn

## 2. Load the dataset

Update `DATA_PATH` if you move the CSV. `low_memory=False` helps pandas infer the mixed product-code and description columns consistently.


In [ ]:
from pathlib import Path

DATA_PATH = Path(r'C:\Users\ASUS\OneDrive\Desktop\internship\task2\online_retail_II.csv')
if not DATA_PATH.exists():
    # If the CSV is beside this notebook, this portable fallback will be used.
    DATA_PATH = Path('online_retail_II.csv')

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')
display(df.head())


## 3. Inspect structure and data quality

Check types, missing values, duplicates, numeric ranges, and invoice prefixes. Invoice numbers beginning with `C` indicate cancellations in this dataset.


In [ ]:
display(df.info())
display(df.describe(include='all').T)
quality = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'distinct_values': df.nunique(dropna=True)
}).sort_values('missing_count', ascending=False)
display(quality)
print('Duplicate rows:', df.duplicated().sum())
print('Invoice examples:', df['Invoice'].astype(str).str[:1].value_counts().head())


## 4. Clean transactions

Trim text columns, parse dates, remove exact duplicate rows, and exclude cancellations and rows that cannot represent a positive purchase. Keep the original `df` available for reference; `sales` contains eligible purchase lines.


In [ ]:
df = df.drop_duplicates().copy()
df['Invoice'] = df['Invoice'].astype('string').str.strip()
df['StockCode'] = df['StockCode'].astype('string').str.strip()
df['Description'] = df['Description'].astype('string').str.strip()
df['Country'] = df['Country'].astype('string').str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['Customer ID'] = pd.to_numeric(df['Customer ID'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

sales = df.dropna(subset=['InvoiceDate', 'Customer ID', 'Invoice', 'Quantity', 'Price']).copy()
sales = sales[~sales['Invoice'].str.upper().str.startswith('C', na=False)]
sales = sales[(sales['Quantity'] > 0) & (sales['Price'] > 0)]
sales['Revenue'] = sales['Quantity'] * sales['Price']
sales['Customer ID'] = sales['Customer ID'].astype('int64')

print(f'Rows retained for RFM: {len(sales):,} of {len(df):,}')
print(f'Customers retained: {sales["Customer ID"].nunique():,}')


## 5. Descriptive statistics

Average purchase value is calculated per invoice (order). Purchase frequency is the number of distinct invoices per customer. Historical customer revenue is used as a simple observed lifetime-value proxy; the data does not include a forward-looking lifetime model.


In [ ]:
invoice_sales = sales.groupby(['Customer ID', 'Invoice'], as_index=False)['Revenue'].sum()
customer_overview = invoice_sales.groupby('Customer ID').agg(
    purchase_frequency=('Invoice', 'nunique'),
    historical_customer_value=('Revenue', 'sum'),
    average_order_value=('Revenue', 'mean')
)
display(customer_overview.describe().T.round(2))
print('Overall average purchase value (per invoice):', round(invoice_sales['Revenue'].mean(), 2))
print('Observed historical customer value (median):', round(customer_overview['historical_customer_value'].median(), 2))


## 6. Build RFM features

Recency is measured in days since each customer's most recent purchase. Frequency counts distinct invoices; Monetary sums eligible line revenue. The snapshot date is one day after the latest transaction.


In [ ]:
snapshot_date = sales['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = sales.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max().normalize()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum')
)
rfm = rfm.replace([np.inf, -np.inf], np.nan).dropna()
display(rfm.describe().round(2))
print('Snapshot date:', snapshot_date.date(), '| Customers:', f'{len(rfm):,}')


## 7. Prepare and standardise features

RFM values are typically right-skewed. Apply `log1p` to reduce skew, then standardise with `StandardScaler` so each measure has comparable influence on Euclidean distance.


In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
rfm_model = rfm[features].copy()
rfm_log = np.log1p(rfm_model)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log)


## 8. Select K with the elbow method

Compare within-cluster sum of squares (inertia) across K values. Look for the point where adding another cluster yields a smaller improvement. The elbow is a decision aid, so review the profiles and plots too.


In [ ]:
k_values = range(2, 11)
inertias = []
silhouettes = []
for k in k_values:
    candidate = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = candidate.fit_predict(X_scaled)
    inertias.append(candidate.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(k_values), inertias, marker='o')
ax[0].set(title='Elbow method', xlabel='Number of clusters (K)', ylabel='Inertia')
ax[1].plot(list(k_values), silhouettes, marker='o', color='darkorange')
ax[1].set(title='Silhouette reference', xlabel='Number of clusters (K)', ylabel='Silhouette score')
plt.tight_layout()
plt.show()
display(pd.DataFrame({'K': list(k_values), 'Inertia': inertias, 'Silhouette': np.round(silhouettes, 3)}))


**Choose K after reviewing the elbow plot.** The default below is 4; change it if the elbow and segment profiles suggest another value.


In [ ]:
K = 4  # Adjust after reviewing the elbow chart
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)


## 9. Profile customer segments

Review cluster means and medians. Lower recency means more recent activity; higher frequency and monetary values mean more orders and spend. The ranking below adds readable segment labels based on the cluster's relative RFM profile.


In [ ]:
profile = rfm.groupby('Cluster')[features].agg(['mean', 'median', 'count']).round(2)
display(profile)

summary = rfm.groupby('Cluster').agg(
    customers=('Recency', 'size'),
    Recency=('Recency', 'mean'),
    Frequency=('Frequency', 'mean'),
    Monetary=('Monetary', 'mean')
).round(2)
summary['customer_share_pct'] = (summary['customers'] / len(rfm) * 100).round(1)
display(summary.sort_values('Monetary', ascending=False))


## 10. Visualise clusters: two feature combinations

Scatter plots use original RFM units for interpretation. Log axes make long-tailed frequency and monetary values easier to compare.


In [ ]:
plot_df = rfm.reset_index()
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=plot_df, x='Recency', y='Frequency', hue='Cluster', palette='tab10', alpha=.65, ax=ax[0])
ax[0].set_yscale('log')
ax[0].set_title('Recency vs Frequency')
sns.scatterplot(data=plot_df, x='Frequency', y='Monetary', hue='Cluster', palette='tab10', alpha=.65, legend=False, ax=ax[1])
ax[1].set_xscale('log'); ax[1].set_yscale('log')
ax[1].set_title('Frequency vs Monetary value')
plt.tight_layout(); plt.show()


## 11. Customers per cluster


In [ ]:
counts = rfm['Cluster'].value_counts().sort_index()
plt.figure(figsize=(8, 4))
sns.barplot(x=counts.index.astype(str), y=counts.values, color='steelblue')
plt.title('Number of customers per cluster')
plt.xlabel('Cluster'); plt.ylabel('Customers')
for i, value in enumerate(counts.values):
    plt.text(i, value, f'{value:,}', ha='center', va='bottom')
plt.tight_layout(); plt.show()


## 12. Segment interpretation and marketing actions

Use the table to connect the data to actions. Cluster numbers are arbitrary and can change when K or the data changes; interpret each row using its RFM values.


In [ ]:
def segment_action(row):
    # These rules are intentionally simple and should be checked against the profile table.
    if row['Recency'] <= summary['Recency'].quantile(0.25) and row['Frequency'] >= summary['Frequency'].median() and row['Monetary'] >= summary['Monetary'].median():
        return 'Loyal high-value: reward with VIP perks, early access, and referral offers.'
    if row['Recency'] > summary['Recency'].quantile(0.75):
        return 'At risk / inactive: run a win-back email with a relevant incentive; suppress if unresponsive.'
    if row['Frequency'] <= summary['Frequency'].median() and row['Monetary'] <= summary['Monetary'].median():
        return 'New or low-engagement: use onboarding, product education, and a second-purchase offer.'
    return 'Growing / regular: recommend complementary products and encourage repeat purchases.'

actions = summary.copy()
actions['Suggested segment action'] = actions.apply(segment_action, axis=1)
display(actions[['customers', 'customer_share_pct', 'Recency', 'Frequency', 'Monetary', 'Suggested segment action']])


## 13. Key findings and next steps

Write 2–4 findings after running the notebook, using the actual cluster profiles and customer counts. For each cluster, state its size, typical recency/frequency/spend, and one marketing action. Consider testing campaigns with a holdout group and tracking incremental revenue.

**Limitations:** RFM is based on historical transactions and does not explain why customers buy. Revenue is not profit; returns and cancellations are excluded from the positive-purchase metrics. The selected K and marketing rules should be reviewed with business context.
